# 4.2 Minimal FastAPI Backend

This notebook explains and tests the Python modules in the `backend` folder. The web server itself is started from a terminal, not from the notebook.

## Folder structure

```text
backend/
├── __init__.py
├── schemas.py
├── predictor.py
├── main.py
├── run.py
└── 4_2_FastAPI_Backend.ipynb
```

- `__init__.py` marks the folder as an explicit Python package.
- `schemas.py` defines the HTTP request and response shapes.
- `predictor.py` owns model loading and prediction logic.
- `main.py` imports those modules and defines the API routes.
- `run.py` is the development entry point for Uvicorn.

When `run.py` is executed directly, Python automatically adds its directory to the module search path. Therefore imports such as `from predictor import IrisPredictor` resolve to `predictor.py` in this folder.

## Import the application in a notebook

Notebook kernels may start in the repository root rather than in the notebook directory. The following setup adds the backend folder to `sys.path` only for this notebook session.

In [ ]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'main.py').exists():
  BACKEND_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'backend').exists():
  BACKEND_DIR = CURRENT_DIR / 'backend'
else:
  BACKEND_DIR = CURRENT_DIR / '04_Solution_4' / 'backend'

if str(BACKEND_DIR) not in sys.path:
  sys.path.insert(0, str(BACKEND_DIR))

print(f'Backend directory: {BACKEND_DIR.resolve()}')

In [ ]:
from fastapi.testclient import TestClient

from main import app

client = TestClient(app)

## Test the health endpoint

In [3]:
response = client.get('/health')
print(response.status_code)
print(response.json())
assert response.status_code == 200
assert response.json() == {'status': 'ok'}

200
{'status': 'ok'}


## Test one prediction

In [4]:
payload = {
  'sepal_length_cm': 5.1,
  'sepal_width_cm': 3.5,
  'petal_length_cm': 1.4,
  'petal_width_cm': 0.2,
}

response = client.post('/predict', json=payload)
print(response.status_code)
print(response.json())
assert response.status_code == 200

200
{'predicted_class': 'setosa', 'probabilities': {'setosa': 0.9808127381969128, 'versicolor': 0.019186992121990206, 'virginica': 2.6968109686921343e-07}}


## Observe automatic validation

Pydantic validates the request before the route calls the predictor. A negative measurement violates the schema and produces status code 422.

In [5]:
invalid_payload = payload | {'petal_width_cm': -1.0}
response = client.post('/predict', json=invalid_payload)
print(response.status_code)
print(response.json())
assert response.status_code == 422

422
{'detail': [{'type': 'greater_than', 'loc': ['body', 'petal_width_cm'], 'msg': 'Input should be greater than 0', 'input': -1.0, 'ctx': {'gt': 0.0}}]}


## Start the API

Open a terminal in this backend folder and run:

```text
python run.py
```

The API is then available at `http://127.0.0.1:8000`. FastAPI's interactive documentation is available at `http://127.0.0.1:8000/docs`.

Keep this terminal running while working with the frontend. Stop the server with `Ctrl+C`.